# 07b · Fine-Tuning — Colab (T4)

GPU execution path for notebook 07b. Same logic, staged on local disk, with
mixed precision on CUDA.

## In Drive before running
```
DFU_project/
  data/interim/folds_infection.csv
  outputs/results_infection.json     <- notebook 07's frozen baseline
  images.zip                         <- built by make_images_zip.py
  07b_finetune.ipynb                 <- this cell runs it
```

If you already ran `05_feature_extraction_colab.ipynb`, `images.zip` and the
staged image folders are the same ones this notebook needs — no need to rebuild.

## Resume
Each fold checkpoints to `outputs/finetune_ckpt/fold{k}_pred.npz`. If this
notebook disconnects partway through, upload whatever checkpoints exist back to
Drive, restore them in Cell 4 below, and rerun — completed folds are skipped.


In [ ]:
# Cell 1 · GPU check
import torch, subprocess
print(subprocess.run(['nvidia-smi','--query-gpu=name','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'no GPU')
assert torch.cuda.is_available(), 'Runtime > Change runtime type > T4 GPU'
print('cuda ready')

In [ ]:
# Cell 2 · mount Drive
from google.colab import drive; drive.mount('/content/drive')
from pathlib import Path
DRIVE = Path('/content/drive/MyDrive/DFU_project')   # edit to match
assert DRIVE.exists(), f'{DRIVE} not found'
print('found', DRIVE)

In [ ]:
# Cell 3 · stage images and CSVs, namespaced by corpus
# Same zip structure as notebook 05: roboflow/ and dfuc/ subfolders, so a
# filename shared between corpora cannot silently overwrite the other.
import shutil, time, zipfile
from pathlib import Path

LOCAL = Path('/content/work'); LOCAL.mkdir(exist_ok=True)
(LOCAL/'data/interim').mkdir(parents=True, exist_ok=True)
(LOCAL/'outputs').mkdir(parents=True, exist_ok=True)

for f in ['data/interim/folds_infection.csv', 'outputs/results_infection.json']:
    src = DRIVE / f
    if src.exists():
        (LOCAL/f).parent.mkdir(parents=True, exist_ok=True)
        shutil.copy(src, LOCAL/f)
        print('copied', f)
    else:
        print('MISSING (optional if not comparing to baseline):', f)

t0 = time.time()
shutil.copy(DRIVE/'images.zip', LOCAL/'images.zip')
with zipfile.ZipFile(LOCAL/'images.zip') as z:
    z.extractall(LOCAL/'images')
n_df = sum(1 for _ in (LOCAL/'images/dfuc').glob('*.jpg'))
print(f'staged {n_df:,} dfuc images in {time.time()-t0:.0f}s')
assert n_df > 0, 'expected images/dfuc/ after extraction'

In [ ]:
# Cell 4 · rewrite paths, restore any prior checkpoints
import pandas as pd
from pathlib import Path

df_index = {p.name: str(p) for p in (LOCAL/'images/dfuc').glob('*.jpg')}
inf = pd.read_csv(LOCAL/'data/interim/folds_infection.csv')
inf['path'] = inf['path'].map(lambda p: df_index.get(Path(str(p)).name))
miss = int(inf.path.isna().sum())
print(f'unmatched: {miss}')
assert miss == 0, 'rebuild images.zip from the current folds_infection.csv'
inf.to_csv(LOCAL/'data/interim/folds_infection.csv', index=False)

# restore checkpoints from a previous interrupted run, if any
ckpt_src = DRIVE/'outputs/finetune_ckpt'
ckpt_dst = LOCAL/'outputs/finetune_ckpt'; ckpt_dst.mkdir(parents=True, exist_ok=True)
if ckpt_src.exists():
    n = 0
    for f in ckpt_src.glob('fold*_pred.npz'):
        shutil.copy(f, ckpt_dst/f.name); n += 1
    print(f'restored {n} fold checkpoints; those folds will be skipped')
else:
    print('no prior checkpoints; starting from fold 0')

In [ ]:
# Cell 5 · run notebook 07b
# Streams output live rather than buffering it. subprocess.run with
# capture_output=True holds ALL stdout until the process exits, so a
# 20-minute training run would show nothing for 20 minutes, then dump
# everything at once. Popen + iterating the stream avoids that.
import subprocess, sys
from pathlib import Path

%cd /content/work
src = DRIVE / '07b_finetune.ipynb'
assert src.exists(), f'{src} not found. Upload 07b_finetune.ipynb to Drive.'
import shutil; shutil.copy(src, 'nb07b.ipynb')

subprocess.run([sys.executable, '-m', 'nbconvert', '--to', 'python',
                'nb07b.ipynb', '--output', 'nb07b'], check=True)
code = Path('nb07b.py').read_text().replace('raise SystemExit(1)', 'pass')
Path('nb07b.py').write_text(code)

print('starting training, output streams live below:\n' + '-'*50)
proc = subprocess.Popen([sys.executable, '-u', 'nb07b.py'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)
lines = []
for line in proc.stdout:
    print(line, end='')
    lines.append(line)
proc.wait()
if proc.returncode != 0:
    raise RuntimeError('nb07b.py failed, see output above for the traceback')

In [ ]:
# Cell 6 · save results and checkpoints back to Drive
import shutil
from pathlib import Path

dst = DRIVE / 'outputs'; dst.mkdir(parents=True, exist_ok=True)
res = LOCAL/'outputs/results_infection_finetuned.json'
if res.exists():
    shutil.copy(res, dst/res.name)
    print('saved', res.name)

ck_dst = dst/'finetune_ckpt'; ck_dst.mkdir(exist_ok=True)
n = 0
for f in (LOCAL/'outputs/finetune_ckpt').glob('*'):
    shutil.copy(f, ck_dst/f.name); n += 1
print(f'saved {n} checkpoint files')
print('Colab wipes /content on disconnect. Always run this before closing.')